- PDF 랜더링 :PyMuPDF(fitz) 사용해서 pdf페이지를 PNG 이미지로 변환 및 저장
- 비전 기반 페이지 요약 : gpt-5.4-nano  각 페이지를 해석해서 표 구조를 포함한 텍스트 요약문을 도출
- 요약-이미지 인덱싱 : 요약문 텍스트는 chromadb 임베딩저장 메타데이터에 원본 이미지 경로를 매핑
- 검색 및 리트리버 : 질문에 맞는 페이지 요약문이 탐지되면 관련 원본 페이지 이미지를 확보
- 멀티모달 답변 생성 : gpt-4.5-nano 질문과 원본페이지 이미지를 동시에 llm에 제공해서 시각적인 데이터(표구조포함)를 직접 해석한 답변을 완성
- 웹검색 Fallback연동 : 내부지식이 부족한 경우 네이버openai/duckduckgo  실시간 웹 검색으로 답변을 보강

### STEP1 환경설정

In [1]:
import os
from dotenv import load_dotenv
load_dotenv(override=True)
os.getenv('OPENAI_API_KEY')[:20]

'sk-proj-FRucyNlpe5gd'

### STEP2  PyMuPDF(fitz)를 이용한 PDF의 고화질 이미지 변환

In [5]:
import fitz
import glob
os.makedirs('./docs_images', exist_ok=True)
pdf_files = glob.glob("./doc/*.pdf")
print('pdf 목록 :',pdf_files)
page_images = []
for pdf in pdf_files:
    print(f'이미지 랜더링 중..{pdf}')
    doc = fitz.open(pdf)    
    base_name = os.path.splitext(os.path.basename(pdf))[0]
    for page_idx in range(len(doc)):
        page = doc.load_page(page_idx)
        pix = page.get_pixmap(dpi=150)
        image_path = f'./docs_images/page_{base_name}_{page_idx}.png'
        pix.save(image_path)
        page_images.append({
            "image_path":image_path,
            "source":pdf,
            "page":page_idx
        })
print(f"총 {len(page_images)}개의 pdf 페이지 이미지 렌더링 완료")
    

pdf 목록 : ['./doc\\01._북한_원산갈마해안관광특별구법에_대한_법적_고찰.pdf', './doc\\02._인도네시아_헌법의_특성과_체계.pdf', './doc\\03._미등록_아동_방지를_위한_출생등록될_권리의_법제화_방안.pdf', './doc\\법제시론_국회_입법기능의_정상화_전학선.pdf']
이미지 랜더링 중.../doc\01._북한_원산갈마해안관광특별구법에_대한_법적_고찰.pdf
이미지 랜더링 중.../doc\02._인도네시아_헌법의_특성과_체계.pdf
이미지 랜더링 중.../doc\03._미등록_아동_방지를_위한_출생등록될_권리의_법제화_방안.pdf
이미지 랜더링 중.../doc\법제시론_국회_입법기능의_정상화_전학선.pdf
총 139개의 pdf 페이지 이미지 렌더링 완료


### step3  GPT-5.4-NANO 비전 모델을 이용한 페이지별 구조화 요약 및 Chroma인덱싱

In [ ]:
import base64
from langchain_core.messages import HumanMessage
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_core.documents import Document

def encode_image(image_path):
    with open(image_path, 'rb') as image_file:
        return base64.b64encode(image_file.read()).decode('utf-8')
print(f'gpt-5.4-nano 비전을 통한 pdf 페이지 해석 및 요약 생성 시작...')    
llm_model = 'gpt-5.4-nano'
llm_vision = ChatOpenAI(model=llm_model,temperature=0,max_completion_tokens=1024)
summarized_docs = []
for item in page_images[:10]:  # 테스트를 위해서 10개 pdf 이미지만 처리
    image_path = item['image_path']
    base64_image = encode_image(image_path)
    
    prompt_message = HumanMessage(
        content=[
            {'type':'text', 
             'text':'pdf 페이지의 이미지를 상세히 요약해 주세요, 페이지 내에 표(Table)나 그래프, 다이그램이 존재한다면'
             ', 그 안의 데이터를 하나도 누락하지 말고 마크다운 테이블 형태(|컬럼1|컬럼2|)로 상세하게 표기해주세요.'
             '시각적인 정보와 텍스트 문맥을 분석하여 정보가 손실되지 않도록 작성해 주세요, 한국어로 작성해 주세요'},
            {'type':'image_url',
             'image_url':{
                 'url': f"data:image/png;base64,{base64_image}"
             }}
        ]
    )
    response = llm_vision.invoke([prompt_message])
    summary_text = response.content
    doc = Document(
        page_content=summary_text,
        metadata = {
            'image_path':image_path,
            'source':item['source'],
            'page':item['page']
        }
    )
    summarized_docs.append(doc)
    print(f' - 요약완료 {image_path} 크기:{len(summary_text)}자')
embeddings = OpenAIEmbeddings(model='text-embedding-3-small')    
persist_dir_vision = './chroma_db_vision'
vectorstore_vision = Chroma.from_documents(
    documents=summarized_docs,
    embedding=embeddings,
    persist_directory=persist_dir_vision,
    collection_name='vision_db'
)
retirver_vision = vectorstore_vision.as_retriever(search_kwargs = {'k':2})
print('비전 RAG용 ChromaDB 구축 완료')

gpt-5.4-nano 비전을 통한 pdf 페이지 해석 및 요약 생성 시작...
...
